# Pré-processamento v2 → `cbf_df1.csv`

Melhorias em relação à v1:
- Coluna `geral` reconstruída do zero via novo join (sem ano)
- Colunas desnecessárias removidas (`tmdb_id`, `imdb_id`, `tmdb_rating`, `tmdb_votes`, `imdb_rating`, `imdb_votes`)
- Título limpo (sem o `(YYYY)`)
- **Pesos por campo**: `genres` e `keywords` repetidos 3×, `director` 2×
- **Lemmatização** (NLTK WordNetLemmatizer)
- **Remoção de tokens curtos** (≤ 2 caracteres)
- Lowercase + remoção de pontuação + remoção de acentos

In [3]:
import pandas as pd
import unicodedata
import re
import nltk

nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

df = pd.read_csv('data/cbf_df.csv')
print('Shape original:', df.shape)
df.columns.tolist()

[nltk_data] Downloading package wordnet to /home/samuel/nltk_data...
[nltk_data] Downloading package omw-1.4 to /home/samuel/nltk_data...


Shape original: (9708, 17)


['movie_id',
 'title',
 'genres',
 'year',
 'tmdb_id',
 'imdb_id',
 'overview',
 'tagline',
 'tmdb_rating',
 'tmdb_votes',
 'imdb_rating',
 'imdb_votes',
 'keywords',
 'cast',
 'director',
 'tags',
 'geral']

## 1. Drop de colunas desnecessárias e limpeza do título

In [4]:
DROP_COLS = ['tmdb_id', 'imdb_id', 'tmdb_rating', 'tmdb_votes', 'imdb_rating', 'imdb_votes', 'geral']
df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

# Remove o "(YYYY)" do título — ex: "Toy Story (1995)" → "Toy Story"
df['title'] = df['title'].str.replace(r'\s*\(\d{4}\)\s*$', '', regex=True).str.strip()

print('Shape após drop:', df.shape)
df[['movie_id', 'title', 'year']].head(5)

Shape após drop: (9708, 10)


,movie_id,title,year
0,1,Toy Story,1995.0
1,2,Jumanji,1995.0
2,3,Grumpier Old Men,1995.0
3,4,Waiting to Exhale,1995.0
4,5,Father of the Bride Part II,1995.0


## 2. Funções de pré-processamento

In [5]:
def normalize_text(text):
    """Lowercase + remove acentos + remove pontuação."""
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = unicodedata.normalize('NFD', text)
    text = ''.join(c for c in text if unicodedata.category(c) != 'Mn')
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def lemmatize_and_filter(text):
    """Lemmatiza e remove tokens com 2 caracteres ou menos."""
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens if len(t) > 2]
    return ' '.join(tokens)


def safe(value):
    """Retorna string limpa ou vazia se NaN."""
    if pd.isna(value):
        return ''
    return str(value).strip()


def pipe_to_space(text):
    """Substitui '|' por espaço (usado em genres e tags)."""
    return safe(text).replace('|', ' ')

## 3. Rebuild da coluna `geral` com pesos por campo

In [6]:
def build_geral(row):
    """
    Reconstrói a coluna geral sem o ano.
    Pesos aplicados pela repetição do campo:
      - genres    × 3  (sinal mais forte de conteúdo)
      - keywords  × 3  (descritores de tema)
      - director  × 2  (estilo autoral relevante)
      - demais    × 1
    """
    title    = safe(row['title'])
    genres   = pipe_to_space(row['genres'])
    overview = safe(row['overview'])
    tagline  = safe(row['tagline'])
    keywords = safe(row['keywords'])
    cast     = safe(row['cast'])
    director = safe(row['director'])
    tags     = pipe_to_space(row['tags'])

    parts = (
        [title]
        + [genres] * 3
        + [overview]
        + [tagline]
        + [keywords] * 3
        + [cast]
        + [director] * 2
        + [tags]
    )

    return ' '.join(p for p in parts if p)


df['geral'] = df.apply(build_geral, axis=1)
df['geral'].head(2)

0    Toy Story Adventure Animation Children Comedy ...
1    Jumanji Adventure Children Fantasy Adventure C...
Name: geral, dtype: str

## 4. Normalização + Lemmatização + Filtro de tokens curtos

In [7]:
df['geral'] = (
    df['geral']
    .apply(normalize_text)       # lowercase + sem acentos + sem pontuação
    .apply(lemmatize_and_filter) # lemmatização + remove tokens <= 2 chars
)

print('Exemplo após processamento:')
print(df['geral'].iloc[0][:300])

Exemplo após processamento:
toy story adventure animation child comedy fantasy adventure animation child comedy fantasy adventure animation child comedy fantasy led woody andy toy live happily his room until andy birthday brings buzz lightyear onto the scene afraid losing his place andy heart woody plot against buzz but when c


## 5. Verificação e salvar

In [8]:
print('Shape final:', df.shape)
print('Colunas:', df.columns.tolist())
print('Vazios em geral:', df['geral'].eq('').sum())
print('NaN em geral   :', df['geral'].isna().sum())
df.head(2)

Shape final: (9708, 11)
Colunas: ['movie_id', 'title', 'genres', 'year', 'overview', 'tagline', 'keywords', 'cast', 'director', 'tags', 'geral']
Vazios em geral: 0
NaN em geral   : 0


,movie_id,title,genres,year,overview,tagline,keywords,cast,director,tags,geral
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995.0,"Led by Woody, Andy's toys live happily in his ...",The adventure takes off when toys come to life!,rescue friendship mission jealousy villain bul...,TomHanks TimAllen DonRickles JimVarney Wallace...,JohnLasseter,pixar|pixar|fun,toy story adventure animation child comedy fan...
1,2,Jumanji,Adventure|Children|Fantasy,1995.0,When siblings Judy and Peter discover an encha...,It's a jungle in here.,giantinsect boardgame disappearance jungle rec...,RobinWilliams KirstenDunst BradleyPierce Bonni...,JoeJohnston,fantasy|magic board game|Robin Williams|game,jumanji adventure child fantasy adventure chil...


In [9]:
df.to_csv('data/cbf_df1.csv', index=False)
print('Salvo em data/cbf_df1.csv')

Salvo em data/cbf_df1.csv
